In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../Datasets/Test_data.csv', encoding='utf-8')
df.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg19.1,Spoštovani poslanke in poslanci! Aprila tega l...,Spoštovani poslanke in poslanci!,Dear Members and Members!,False,P_Neutral,3.0,3.770256,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...
1,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.1,Spoštovani poslanke in poslanci! Aprila tega l...,Aprila tega leta je prišlo do pozebe do vremen...,"In April of this year, there was a cold of wea...",False,P_Neutral,3.0,1.725317,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...
2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.2,Spoštovani poslanke in poslanci! Aprila tega l...,"Na ministrstvu in na Vladi, skupaj z nevladnim...","At the Ministry and in the Government, togethe...",False,P_Neutral,3.0,3.962013,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...
3,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.3,Spoštovani poslanke in poslanci! Aprila tega l...,Ker je treba z izvajanjem prvega ukrepa začeti...,Since the implementation of the first measure ...,False,P_Neutral,3.0,3.028064,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...
4,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.4,Spoštovani poslanke in poslanci! Aprila tega l...,Hvala.,Thank you.,False,P_Neutral,3.0,4.510695,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...


## Dataset preparation

### Adding heuristics scores
Adding aggregated scores for the following heuristics: 
- mean
- median
- sentence count (average based on no. of sentences within utterance)
- word count (average based on no. of words within individual sentences)
- character count (average based on no. of characters within individual sentences)
- position-based average
- sentiment-intensity average

In [3]:
aggregation = df.groupby('ID')['annotation_sent'].agg(
    mean = 'mean',
    median = 'median'
)
df = df.merge(aggregation, on='ID')

In [4]:
def count_avg(df: pd.DataFrame, id_col: str, sent_col: str, annotation_col: str) -> pd.DataFrame:
    df.loc[:, 'count'] = df.groupby(id_col)[sent_col].transform('count')
    count_avg = (
        df.groupby(id_col, group_keys=True)
        .apply(lambda x: (x[annotation_col] * x['count']).sum() / x['count'].sum())
        .reset_index(name='count_avg')
    )
    df = df.merge(count_avg, on=id_col, how='left')
    return df

def word_avg(df: pd.DataFrame, id_col: str, sent_col: str, annotation_col: str) -> pd.DataFrame:
    df["words"] = df["text_sent"].apply(lambda x: len(x.split()))
    word_avg = (
        df.groupby("ID")
        .apply(lambda group: (group["annotation_sent"] * group["words"]).sum() / group["words"].sum())
        .reset_index(name="word_avg")
        )
    df = df.merge(word_avg, on=id_col, how='left')
    return df

def char_avg(df: pd.DataFrame, id_col: str, sent_col: str, annotation_col: str) -> pd.DataFrame:
    df["char_length"] = df["text_sent"].apply(len)
    char_avg = (
        df.groupby('ID')
        .apply(lambda x: (x["annotation_sent"] * x["char_length"]).sum() / x["char_length"].sum())
        .reset_index(name='char_avg')
    )
    df = df.merge(char_avg, on='ID')
    return df



In [5]:
df = count_avg(df, id_col='ID', sent_col='sent_id', annotation_col='annotation_sent')
df = word_avg(df, id_col='ID', sent_col='sent_id', annotation_col='annotation_sent')
df = char_avg(df, id_col='ID', sent_col='sent_id', annotation_col='annotation_sent')

df

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/3846160506.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(id_col, group_keys=True)
/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/3846160506.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("ID")
/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/3846160506.py:24: DeprecationWarning: 

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,mean,median,count,count_avg,words,word_avg,char_length,char_avg
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg19.1,Spoštovani poslanke in poslanci! Aprila tega l...,Spoštovani poslanke in poslanci!,Dear Members and Members!,False,P_Neutral,3.0,3.770256,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...,3.399269,3.770256,5,3.399269,4,3.054111,32,3.147832
1,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.1,Spoštovani poslanke in poslanci! Aprila tega l...,Aprila tega leta je prišlo do pozebe do vremen...,"In April of this year, there was a cold of wea...",False,P_Neutral,3.0,1.725317,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...,3.399269,3.770256,5,3.399269,20,3.054111,119,3.147832
2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.2,Spoštovani poslanke in poslanci! Aprila tega l...,"Na ministrstvu in na Vladi, skupaj z nevladnim...","At the Ministry and in the Government, togethe...",False,P_Neutral,3.0,3.962013,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...,3.399269,3.770256,5,3.399269,25,3.054111,189,3.147832
3,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.3,Spoštovani poslanke in poslanci! Aprila tega l...,Ker je treba z izvajanjem prvega ukrepa začeti...,Since the implementation of the first measure ...,False,P_Neutral,3.0,3.028064,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...,3.399269,3.770256,5,3.399269,17,3.054111,106,3.147832
4,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.seg20.4,Spoštovani poslanke in poslanci! Aprila tega l...,Hvala.,Thank you.,False,P_Neutral,3.0,4.510695,{'Text_ID': 'ParlaMint-SI-en_2016-07-11-SDZ7-R...,3.399269,3.770256,5,3.399269,1,3.054111,6,3.147832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2692,ParlaMint-SI_2015-05-13-SDZ7-Izredna-13.ana.u1,ParlaMint-SI_2015-05-13-SDZ7-Izredna-13.ana.se...,Spoštovani kolegice poslanke in kolegi poslanc...,"Predsednik Vlade je predlagal Državnemu zboru,...",The President of the Government proposed to th...,True,P_Neutral,3.0,2.879747,{'Text_ID': 'ParlaMint-SI-en_2015-05-13-SDZ7-I...,3.118520,3.113337,22,3.118520,23,2.862954,151,2.881605
2693,ParlaMint-SI_2015-05-13-SDZ7-Izredna-13.ana.u1,ParlaMint-SI_2015-05-13-SDZ7-Izredna-13.ana.se...,Spoštovani kolegice poslanke in kolegi poslanc...,Besedo dajem predsedniku Vlade dr. Miru Cerarj...,I give the floor to the President of the Gover...,True,P_Neutral,3.0,2.952675,{'Text_ID': 'ParlaMint-SI-en_2015-05-13-SDZ7-I...,3.118520,3.113337,22,3.118520,13,2.862954,97,2.881605
2694,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...,3.872611,3.854102,3,3.872611,2,3.889854,11,3.881988
2695,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,Besedo ima mag. Branko Grims.,The word has magic. Branko Grimes.,True,P_Neutral,3.0,3.854102,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...,3.872611,3.854102,3,3.872611,5,3.889854,29,3.881988


### Feature building

In [6]:
columns = ['ID', 'annotation_sent', 'annotation_utt_score', 'count', 'words', 'char_length', 'mean', 'median', 'count_avg', 'word_avg', 'char_avg']
df = df[columns]

df['sum'] = df.groupby('ID')['char_length'].transform('sum')
df['Q1'] = df.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.25))
df['Q3'] = df.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.75))
df = df.drop(columns=['char_length'])
df.head()

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/2908039945.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sum'] = df.groupby('ID')['char_length'].transform('sum')
/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/2908039945.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Q1'] = df.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.25))
/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4281/2908039945.py:6: SettingWithCopyWarning: 
A valu

,ID,annotation_sent,annotation_utt_score,count,words,mean,median,count_avg,word_avg,char_avg,sum,Q1,Q3
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.770256,3.0,5,4,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013
1,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,1.725317,3.0,5,20,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013
2,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.962013,3.0,5,25,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013
3,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.028064,3.0,5,17,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013
4,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,4.510695,3.0,5,1,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013


In [7]:
data = df.drop_duplicates(subset=['ID']).reset_index(drop=True)
data.head()

,ID,annotation_sent,annotation_utt_score,count,words,mean,median,count_avg,word_avg,char_avg,sum,Q1,Q3
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.770256,3.0,5,4,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013
1,ParlaMint-SI_2010-06-14-SDZ5-Redna-18.ana.u213,3.112779,2.0,1,3,3.112779,3.112779,3.112779,3.112779,3.112779,20,3.112779,3.112779
2,ParlaMint-SI_2008-07-10-SDZ4-Redna-41.ana.u147,3.785013,0.0,25,3,2.227396,2.282060,2.227396,1.992312,2.010223,3639,1.553671,3.244193
3,ParlaMint-SI_2006-10-23-SDZ4-Redna-21.ana.u178,4.084590,3.0,4,2,3.591495,3.661278,3.591495,3.408713,3.379065,114,3.443831,3.808942
4,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.ana.u80,3.625696,5.0,21,7,2.542785,2.987792,2.542785,2.578488,2.614748,2848,0.486142,4.084590


## Applying RF model

In [8]:
import joblib

model = joblib.load('../Models/RF_model.pkl')

In [9]:
# Features (count, mean, median, sum, character-based average, Q1, Q2)
X_test = data[['count', 'sum', 'mean', 'median', 'char_avg', 'Q1', 'Q3']]
y_test = data['annotation_utt_score']

print(X_test)
print(y_test)

     count   sum      mean    median  char_avg        Q1        Q3
0        5   452  3.399269  3.770256  3.147832  3.028064  3.962013
1        1    20  3.112779  3.112779  3.112779  3.112779  3.112779
2       25  3639  2.227396  2.282060  2.010223  1.553671  3.244193
3        4   114  3.591495  3.661278  3.379065  3.443831  3.808942
4       21  2848  2.542785  2.987792  2.614748  0.486142  4.084590
..     ...   ...       ...       ...       ...       ...       ...
195      7   761  2.211334  3.299057  1.927558  0.159343  3.892215
196      7  1148  2.567764  2.734125  1.869336  1.368551  4.041500
197     25  3616  3.211891  3.033014  3.239146  2.906732  3.494179
198     22  2245  3.118520  3.113337  2.881605  2.883214  3.253081
199      3    47  3.872611  3.854102  3.881988  3.766621  3.969346

[200 rows x 7 columns]
0      3.0
1      2.0
2      0.0
3      3.0
4      5.0
      ... 
195    0.0
196    0.0
197    5.0
198    3.0
199    3.0
Name: annotation_utt_score, Length: 200, dtype: flo

In [10]:
y_pred = model.predict(X_test)
data['predictions'] = y_pred
data

,ID,annotation_sent,annotation_utt_score,count,words,mean,median,count_avg,word_avg,char_avg,sum,Q1,Q3,predictions
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.770256,3.0,5,4,3.399269,3.770256,3.399269,3.054111,3.147832,452,3.028064,3.962013,3.03
1,ParlaMint-SI_2010-06-14-SDZ5-Redna-18.ana.u213,3.112779,2.0,1,3,3.112779,3.112779,3.112779,3.112779,3.112779,20,3.112779,3.112779,2.55
2,ParlaMint-SI_2008-07-10-SDZ4-Redna-41.ana.u147,3.785013,0.0,25,3,2.227396,2.282060,2.227396,1.992312,2.010223,3639,1.553671,3.244193,0.74
3,ParlaMint-SI_2006-10-23-SDZ4-Redna-21.ana.u178,4.084590,3.0,4,2,3.591495,3.661278,3.591495,3.408713,3.379065,114,3.443831,3.808942,2.99
4,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.ana.u80,3.625696,5.0,21,7,2.542785,2.987792,2.542785,2.578488,2.614748,2848,0.486142,4.084590,2.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ParlaMint-SI_2014-03-06-SDZ6-Redna-22.ana.u272,0.033112,0.0,7,24,2.211334,3.299057,2.211334,1.909036,1.927558,761,0.159343,3.892215,0.23
196,ParlaMint-SI_2009-04-22-SDZ5-Redna-05.ana.u226,3.998410,0.0,7,6,2.567764,2.734125,2.567764,1.928471,1.869336,1148,1.368551,4.041500,0.76
197,ParlaMint-SI_2017-02-13-SDZ7-Redna-27.ana.u134,3.608801,5.0,25,5,3.211891,3.033014,3.211891,3.258280,3.239146,3616,2.906732,3.494179,4.01
198,ParlaMint-SI_2015-05-13-SDZ7-Izredna-13.ana.u1,4.078775,3.0,22,9,3.118520,3.113337,3.118520,2.862954,2.881605,2245,2.883214,3.253081,2.79


In [24]:
test_columns = ['ID', 'annotation_utt_score', 'mean', 'median', 'count_avg', 'word_avg', 'char_avg', 'predictions']
test = data[test_columns]
test.to_csv('../Datasets/Test_results/Results.csv', encoding='utf-8', index=False)
test.head()

,ID,annotation_utt_score,mean,median,count_avg,word_avg,char_avg,predictions
0,ParlaMint-SI_2016-07-11-SDZ7-Redna-21.ana.u2,3.0,3.399269,3.770256,3.399269,3.054111,3.147832,3.03
1,ParlaMint-SI_2010-06-14-SDZ5-Redna-18.ana.u213,2.0,3.112779,3.112779,3.112779,3.112779,3.112779,2.55
2,ParlaMint-SI_2008-07-10-SDZ4-Redna-41.ana.u147,0.0,2.227396,2.282060,2.227396,1.992312,2.010223,0.74
3,ParlaMint-SI_2006-10-23-SDZ4-Redna-21.ana.u178,3.0,3.591495,3.661278,3.591495,3.408713,3.379065,2.99
4,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.ana.u80,5.0,2.542785,2.987792,2.542785,2.578488,2.614748,2.64


In [25]:
from sklearn.metrics import mean_absolute_error

mae_mean = mean_absolute_error(test['annotation_utt_score'], test['mean'])
mae_median = mean_absolute_error(test['annotation_utt_score'], test['median'])
mae_count = mean_absolute_error(test['annotation_utt_score'], test['count_avg'])
mae_word = mean_absolute_error(test['annotation_utt_score'], test['word_avg'])
mae_char = mean_absolute_error(test['annotation_utt_score'], test['char_avg'])
mae_rf = mean_absolute_error(test['annotation_utt_score'], test['predictions'])



print('Mean MAE:', mae_mean)
print('Median MAE:', mae_median)
print('Count-weighted avg MAE:', mae_count)
print('Word-weighted avg. MAE:', mae_word)
print('Character-weighted avg. MAE:', mae_char)
print('RF predictions MAE:', mae_rf)


Mean MAE: 1.1120379344939375
Median MAE: 1.0882094806432725
Count-weighted avg MAE: 1.1120379344939375
Word-weighted avg. MAE: 0.9286460126015857
Character-weighted avg. MAE: 0.9262012992458852
RF predictions MAE: 0.4457


In [26]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

def calculate_mae(test: pd.DataFrame) -> pd.DataFrame:
    mae_results = {
        'Model': ['Mean', 'Median', 'Count-weighted avg', 'Word-weighted avg', 'Character-weighted avg', 'Random Forest'],
        'MAE': [
            mean_absolute_error(test['annotation_utt_score'], test['mean']),
            mean_absolute_error(test['annotation_utt_score'], test['median']),
            mean_absolute_error(test['annotation_utt_score'], test['count_avg']),
            mean_absolute_error(test['annotation_utt_score'], test['word_avg']),
            mean_absolute_error(test['annotation_utt_score'], test['char_avg']),
            mean_absolute_error(test['annotation_utt_score'], test['predictions'])
        ]
    }
    
    mae_df = pd.DataFrame(mae_results)
    
    return mae_df



In [27]:
mae_df = calculate_mae(test)
mae_df.to_csv('../Datasets/Test_results/MAE_results.csv', encoding='utf-8', index=False)
mae_df

,Model,MAE
0,Mean,1.112038
1,Median,1.088209
2,Count-weighted avg,1.112038
3,Word-weighted avg,0.928646
4,Character-weighted avg,0.926201
5,Random Forest,0.445700
